# Varmestråling i et rom med glassvegg

## Formfaktorer, radiositet og temperaturdynamikk

### Pilotprosjekt for Matematikk 1

Et rom kan oppleves kaldt selv om lufttemperaturen er tilfredsstillende. En viktig årsak er varmestråling mellom mennesker og kalde overflater, særlig store glassflater.

I dette prosjektet modellerer vi et rektangulært rom der hele frontveggen er glass. Venstre og høyre sidevegg er identiske og slås sammen ved symmetri. Vi beholder gulv og tak som separate flater fordi de vanligvis har ulike temperaturer, materialer og termiske randbetingelser.

Prosjektet har tre hoveddeler:

1. **Formfaktormatrisen:** geometrisk kobling mellom romflatene
2. **Stasjonær radiositet:** direkte og iterativ løsning av et lineært system
3. **Temperaturdynamikk:** et ODE-system der radiositetssystemet løses i hvert Euler-steg

### Læringsmål

Etter prosjektet skal du kunne

- forklare hva en formfaktor betyr,
- bruke summasjons- og resiprositetsreglene,
- redusere et symmetrisk seksflatesystem til fem ukjente,
- bygge en radiositetsmatrise,
- løse et lineært system med `np.linalg.solve`,
- tolke en matriseiterasjon som gjentatte refleksjoner,
- kontrollere energi- og matrisebalanser,
- skrive en termisk energibalanse som et ODE-system,
- bruke Euler når et lineært system må løses i hvert tidssteg.

### Modellavgrensning

Dette er en pedagogisk grå-diffus overflatemodell. Luftens deltakelse i strålingen, speilrefleksjon, solstråling gjennom glasset, temperaturvariasjon innen hver flate og detaljert varmestrøm i konstruksjonene er utelatt i hovedmodellen.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sigma = 5.670374419e-8  # Stefan-Boltzmann-konstant, W/(m^2 K^4)

# Romgeometrien

Rommet har lengde $L$, bredde $B$ og høyde $H$.

De seks fysiske flatene er

1. gulv $G$,
2. tak $T$,
3. venstre vegg $L_v$,
4. høyre vegg $R_v$,
5. bakvegg $B_v$,
6. glassvegg $V$.

På grunn av speilsymmetri setter vi

$$J_{L_v}=J_{R_v}=J_S,$$

og bruker fem radiositetsukjente:

$$
\boxed{
J=
\begin{pmatrix}
J_G\\J_T\\J_S\\J_B\\J_V
\end{pmatrix}.}
$$

Her representerer $S$ de to identiske sideveggene.

In [ ]:
L_rom = 5.0
B_rom = 4.0
H_rom = 2.8

A_G = L_rom*B_rom
A_T = L_rom*B_rom
A_side_en = L_rom*H_rom
A_S = 2*A_side_en
A_B = B_rom*H_rom
A_V = B_rom*H_rom

areal = np.array([A_G, A_T, A_S, A_B, A_V])
navn = ["gulv", "tak", "sidevegger", "bakvegg", "glassvegg"]

for n, A in zip(navn, areal):
    print(f"{n:12s}: {A:5.2f} m^2")

# Del A: Formfaktormatrisen

## A.1 Hva er en formfaktor?

Formfaktoren $F_{ij}$ er andelen av strålingen som forlater flate $i$, og som treffer flate $j$ direkte.

For en lukket innhegning gjelder summasjonsregelen

$$
\boxed{\sum_jF_{ij}=1.}
$$

Formfaktorene oppfyller også resiprositetsregelen

$$
\boxed{A_iF_{ij}=A_jF_{ji}.}
$$

For en plan flate er egenformfaktoren null:

$$F_{ii}=0.$$

I den reduserte modellen er det ett viktig unntak i notasjonen: $F_{SS}$ er ikke en fysisk veggs egenformfaktor. Den representerer stråling fra den venstre sideveggen til den høyre sideveggen.

## A.2 To standardgeometrier

For det rektangulære rommet trenger vi i hovedsak

- parallelle, direkte motstående rektangler,
- vinkelrette rektangler med felles kant.

I denne første piloten gis to ferdige hjelpefunksjoner. Studentene skal forstå hvilke dimensjoner som settes inn og kontrollere resultatene, men trenger ikke utlede de lukkede formlene fra dobbeltintegralet.

In [ ]:
def F_parallel_like(a, b, avstand):
    """Formfaktor mellom like, sentrerte og parallelle rektangler a x b."""
    X = a/avstand
    Y = b/avstand

    ledd_log = 0.5*np.log(
        ((1 + X**2)*(1 + Y**2))/(1 + X**2 + Y**2)
    )
    ledd_x = X*np.sqrt(1 + Y**2)*np.arctan(X/np.sqrt(1 + Y**2))
    ledd_y = Y*np.sqrt(1 + X**2)*np.arctan(Y/np.sqrt(1 + X**2))
    korreksjon = X*np.arctan(X) + Y*np.arctan(Y)

    return 2*(ledd_log + ledd_x + ledd_y - korreksjon)/(np.pi*X*Y)


def F_vinkelrett_felles_kant(a, b, c):
    """
    Flate 1 har areal a*b. Flate 2 har areal a*c.
    Felles kant har lengde a. Returnerer F_1_til_2.
    """
    W = b/a
    H = c/a

    ledd_vinkel = (
        W*np.arctan(1/W)
        + H*np.arctan(1/H)
        - np.sqrt(W**2 + H**2)*np.arctan(1/np.sqrt(W**2 + H**2))
    )

    faktor_1 = ((1 + W**2)*(1 + H**2))/(1 + W**2 + H**2)
    faktor_2 = (
        W**2*(1 + W**2 + H**2)/((1 + W**2)*(W**2 + H**2))
    )**(W**2)
    faktor_3 = (
        H**2*(1 + H**2 + W**2)/((1 + H**2)*(H**2 + W**2))
    )**(H**2)

    ledd_log = 0.25*np.log(faktor_1*faktor_2*faktor_3)
    return (ledd_vinkel + ledd_log)/(np.pi*W)

## Oppgave A1: Grunnleggende kontroller

Beregn formfaktorene mellom

- gulv og tak,
- venstre og høyre sidevegg,
- bakvegg og glassvegg.

Kontroller at alle ligger mellom 0 og 1. Forklar hvilken dimensjon som skal brukes som normalavstand i hvert tilfelle.

In [ ]:
F_GT = ...
F_LR = ...
F_BV = ...

print("F_G->T =", F_GT)
print("F_L->R =", F_LR)
print("F_B->V =", F_BV)

## A.3 Vinkelrette naboflater

Eksempel: gulvet og én sidevegg deler en kant med lengde $L$. Gulvet strekker seg $B$ ut fra kanten, mens veggen strekker seg $H$ opp fra kanten.

Dermed beregnes

$$F_{G,L_v}$$

med

$$a=L,\qquad b=B,\qquad c=H.$$

Den motsatte faktoren finnes sikrest med resiprositet:

$$F_{L_v,G}=\frac{A_G}{A_{L_v}}F_{G,L_v}.$$

## Oppgave A2: Beregn de unike nabofaktorene

Beregn

- gulv til én sidevegg,
- gulv til bakvegg,
- gulv til glassvegg,
- tak til én sidevegg,
- tak til bakvegg,
- tak til glassvegg,
- én sidevegg til bakvegg,
- én sidevegg til glassvegg.

Noen av disse er geometrisk like. Bruk symmetrien før du regner alt på nytt.

In [ ]:
F_G_L = ...
F_G_B = ...
F_G_V = ...

F_T_L = ...
F_T_B = ...
F_T_V = ...

F_L_B = ...
F_L_V = ...

print(F_G_L, F_G_B, F_G_V)
print(F_T_L, F_T_B, F_T_V)
print(F_L_B, F_L_V)

## A.4 Den reduserte femflatematrisen

Når en vanlig flate ser begge sideveggene, summeres bidragene:

$$F_{G,S}=F_{G,L_v}+F_{G,R_v}=2F_{G,L_v}.$$

Fra den representative sideveggen skal vi ikke doble:

$$F_{S,G}=F_{L_v,G}.$$

Sideveggparets diagonalelement er

$$F_{S,S}=F_{L_v,R_v}.$$

Den reduserte matrisen har strukturen

$$
F=
\begin{pmatrix}
0&F_{GT}&2F_{GL}&F_{GB}&F_{GV}\\
F_{TG}&0&2F_{TL}&F_{TB}&F_{TV}\\
F_{LG}&F_{LT}&F_{LR}&F_{LB}&F_{LV}\\
F_{BG}&F_{BT}&2F_{BL}&0&F_{BV}\\
F_{VG}&F_{VT}&2F_{VL}&F_{VB}&0
\end{pmatrix}.
$$

## Oppgave A3: Bruk resiprositet og bygg matrisen

Fyll inn matrisen. Bruk resiprositet for alle motsatte retninger.

I den reduserte modellen må du bruke det samlede sideveggarealet $A_S=2A_{L_v}$ i den aggregerte resiprositetskontrollen.

In [ ]:
F = np.zeros((5, 5))
G, T, S, BAK, V = range(5)

# Kjente retninger fra geometriformlene.
F[G, T] = ...
F[G, S] = ...
F[G, BAK] = ...
F[G, V] = ...

F[T, S] = ...
F[T, BAK] = ...
F[T, V] = ...

F[S, S] = ...
F[S, BAK] = ...
F[S, V] = ...

F[BAK, V] = ...

# Bruk resiprositet for motsatte retninger.
# Eksempel:
F[T, G] = areal[G]/areal[T] * F[G, T]

# Fyll inn resten.

print("Redusert formfaktormatrise:
", F)

## Oppgave A4: Kontroller matrisen

Kontroller

$$F\mathbf 1=\mathbf 1$$

og den arealvektede resiprositeten

$$A_iF_{ij}=A_jF_{ji}.$$

Skriv ut største avvik. Dersom kontrollene ikke holder, undersøk om en faktor er brukt i feil retning eller doblet feil.

In [ ]:
radsummer = ...
resiprositetsfeil = ...

print("Radsummer:", radsummer)
print("Største resiprositetsfeil:", np.max(np.abs(resiprositetsfeil)))
print("Minste og største matriseelement:", np.min(F), np.max(F))

## Lærermerknad om avrunding

Dersom formfaktorene hentes fra tabeller med få sifre, vil radsummer og resiprositet ikke bli eksakte. I piloten kan noen manglende elementer eventuelt bestemmes fra radsummen, men dette bør gjøres systematisk og dokumenteres.

Ikke sett små negative verdier til null uten først å undersøke om det skyldes feil geometri, feil retning eller grov avrunding.

# Del B: Stasjonær radiositet

## B.1 Emisjon og refleksjon

Radiositeten $J_i$ er all stråling som forlater flate $i$ per areal. Den består av

- stråling flaten selv emitterer,
- reflektert innkommende stråling.

For en grå, diffus flate:

$$
J_i
=
\varepsilon_i\sigma T_i^4
+(1-\varepsilon_i)\sum_jF_{ij}J_j.
$$

Definer

$$
E=\operatorname{diag}(\varepsilon_1,\ldots,\varepsilon_5),
\qquad
R=I-E.
$$

Da får vi det lineære systemet

$$
\boxed{
\left[I-RF\right]J
=
E\sigma T^{[4]}.}
$$

## B.2 Overflatetemperaturer og emissiviteter

Bruk Kelvin i Stefan–Boltzmann-loven.

Pilotverdiene er:

- gulv: $24^\circ$C,
- tak: $21^\circ$C,
- sidevegger: $21^\circ$C,
- bakvegg: $21^\circ$C,
- innvendig glassflate: $12^\circ$C.

Emissivitetene gis som modellparametre.

In [ ]:
T_C = np.array([24.0, 21.0, 21.0, 21.0, 12.0])
T_K = T_C + 273.15

emissivitet = np.array([0.92, 0.90, 0.90, 0.90, 0.84])
Eps = np.diag(emissivitet)
Rho = np.eye(5) - Eps

E_emit = Eps @ (sigma*T_K**4)
A_rad = np.eye(5) - Rho @ F

print("Egen emisjon, W/m^2:", E_emit)
print("Radiositetsmatrise:
", A_rad)

## Oppgave B1: Direkte løsning

Løs radiositetssystemet med `np.linalg.solve`.

Beregn deretter innstrålingen

$$G=FJ,$$

netto utgående strålingsfluks

$$q''=J-G,$$

og total netto varme fra hver overflate

$$Q_i=A_iq_i''.$$

In [ ]:
J_direkte = ...
G_inn = ...
q_netto = ...
Q_netto = ...

for navn_i, Ji, Gi, Qi in zip(navn, J_direkte, G_inn, Q_netto):
    print(f"{navn_i:12s}: J={Ji:8.2f}, G={Gi:8.2f}, Q={Qi:9.2f} W")

print("Energibalanse, sum Q:", np.sum(Q_netto), "W")

## Oppgave B2: Tolk fortegnene

1. Hvilke flater avgir netto strålingsvarme?
2. Hvilken flate mottar mest?
3. Hvorfor får den kalde glassveggen et annet fortegn enn de varme romflatene?
4. Hvor nær null er den totale energibalansen?

# B.3 Refleksjon for refleksjon

Den samme radiositetsligningen kan skrives

$$
J=E_{emit}+RFJ.
$$

Dette foreslår iterasjonen

$$
\boxed{
J^{(k+1)}=E_{emit}+RFJ^{(k)}.}
$$

Start med

$$J^{(0)}=E_{emit}.$$

Da er

- $J^{(0)}$: bare egen emisjon,
- første tillegg: bidrag etter én diffus refleksjon,
- neste tillegg: bidrag etter to refleksjoner,
- og så videre.

Vi følger gjennomsnittlige energiflukser, ikke individuelle lysstråler.

## Oppgave B3: Beregn de første refleksjonene

Beregn og skriv ut $J^{(0)}$, $J^{(1)}$, $J^{(2)}$ og $J^{(3)}$.

Sammenlign med den direkte løsningen.

In [ ]:
J0 = E_emit.copy()
J1 = ...
J2 = ...
J3 = ...

for k, Jk in enumerate([J0, J1, J2, J3]):
    print(f"k={k}, J={Jk}")
    print("feil mot direkte løsning:", np.linalg.norm(Jk - J_direkte))

## B.4 Hvert nytt refleksjonsbidrag

Det er enda mer fysisk å lagre bare det nye bidraget:

$$r^{(0)}=E_{emit},$$

$$
\boxed{r^{(k+1)}=RFr^{(k)}.}
$$

Den summerte radiositeten etter $k$ refleksjonsordener er

$$J^{(k)}=\sum_{m=0}^kr^{(m)}.$$

In [ ]:
antall = 25
bidrag = E_emit.copy()
J_sum = bidrag.copy()

bidrag_historikk = [bidrag.copy()]
J_historikk = [J_sum.copy()]

for k in range(antall):
    bidrag = ...
    J_sum = ...
    bidrag_historikk.append(bidrag.copy())
    J_historikk.append(J_sum.copy())

bidrag_historikk = np.array(bidrag_historikk)
J_historikk = np.array(J_historikk)

bidragsnorm = np.linalg.norm(bidrag_historikk, axis=1)
feil = np.linalg.norm(J_historikk - J_direkte, axis=1)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].semilogy(bidragsnorm, "o-")
ax[0].set_xlabel("Refleksjonsorden")
ax[0].set_ylabel("Norm av nytt bidrag")
ax[0].grid()

ax[1].semilogy(feil, "o-")
ax[1].set_xlabel("Iterasjon")
ax[1].set_ylabel("Feil mot direkte løsning")
ax[1].grid()
plt.show()

## Oppgave B4: Et enkelt stoppkriterium

Implementer iterasjonen til

$$\|J^{(k+1)}-J^{(k)}\|<\text{toleranse},$$

eller til et maksimalt antall iterasjoner er nådd.

Sammenlign det beregnede svaret med den direkte løsningen.

In [ ]:
def radiositet_iterativ(F, emissivitet, temperatur_K,
                         toleranse=1e-8, maks_iterasjoner=10000):
    n = len(temperatur_K)
    Eps = np.diag(emissivitet)
    Rho = np.eye(n) - Eps
    E_emit = Eps @ (sigma*temperatur_K**4)

    J = E_emit.copy()

    for k in range(maks_iterasjoner):
        J_ny = ...

        if ...:
            return J_ny, k + 1

        J = J_ny

    raise RuntimeError("Iterasjonen nådde ikke toleransen")

J_iter, antall_iter = radiositet_iterativ(F, emissivitet, T_K)
print("Antall iterasjoner:", antall_iter)
print("Forskjell fra direkte løsning:", np.linalg.norm(J_iter-J_direkte))

## Valgfri numerikkfordypning: Spektralradius

Iterasjonsmatrisen er

$$B=RF.$$

En sentral konvergensbetingelse er

$$\rho(B)<1,$$

hvor $\rho(B)$ er største absoluttverdi av egenverdiene.

Dette er en bro til emnet i numeriske metoder og trenger ikke inngå i vurderingen i Matematikk 1.

In [ ]:
B_iter = Rho @ F
egenverdier_B = np.linalg.eigvals(B_iter)
spektralradius = np.max(np.abs(egenverdier_B))

print("Egenverdier til RF:", egenverdier_B)
print("Spektralradius:", spektralradius)

## Oppgave B5: Emissivitet og konvergens

Sammenlign minst tre scenarier:

1. høy emissivitet på alle flater,
2. pilotverdiene,
3. en hypotetisk lavemissiv glassvegg.

Registrer antall iterasjoner og spektralradius dersom du tar fordypningen.

Forklar fysisk hvorfor høy refleksjonsgrad gir flere viktige refleksjonsordener.

# Del C: Temperaturdynamikk

## C.1 Tre dynamiske temperaturer

Radiositetsmodellen har fem flater, men alle trenger ikke ha egne dynamiske temperaturer.

Vi lar

$$
x(t)=
\begin{pmatrix}
T_{luft}\\T_{gulv}\\T_{glass}
\end{pmatrix}
$$

være dynamiske tilstander. Tak, sidevegger og bakvegg holdes på en gitt temperatur i hovedmodellen.

Ved hvert tidssteg:

1. sett sammen de fem overflatetemperaturene,
2. løs det lineære radiositetssystemet,
3. beregn netto strålingsvarme,
4. beregn temperaturderivertene,
5. gjør Euler-steget.

## C.2 Forenklede energibalanser

Vi bruker

$$
C_l\dot T_l
=
Q_{varme}
-h_{la}A_{rom}(T_l-T_{ute})
-h_{lg}A_G(T_l-T_G)
-h_{lv}A_V(T_l-T_V),
$$

$$
C_G\dot T_G
=
Q_{gulv}
+h_{lg}A_G(T_l-T_G)
-Q_{rad,G},
$$

$$
C_V\dot T_V
=
U_VA_V(T_{ute}-T_V)
+h_{lv}A_V(T_l-T_V)
-Q_{rad,V}.
$$

Fortegnet til $Q_{rad,i}$ er positivt når flate $i$ avgir netto strålingsvarme.

Dette er en pedagogisk modell, ikke en full bygningssimulator.

In [ ]:
# Termiske parametre
C_luft = 1.2e5
C_gulv = 3.0e6
C_glass = 4.0e5

h_tap = 80.0
h_luft_gulv = 2.5
h_luft_glass = 3.0
U_glass = 1.2

T_faste_C = {
    "tak": 21.0,
    "side": 21.0,
    "bak": 21.0
}


def T_ute_C(t):
    return 0.0 if t < 6*3600 else -5.0


def Q_romvarme(t):
    return 1800.0


def Q_gulvvarme(t):
    return 1000.0

## Oppgave C1: Lag en radiositetsfunksjon

Funksjonen skal ta fem overflatetemperaturer i Kelvin og returnere

- radiositetene,
- total netto strålingsvarme fra hver flate.

In [ ]:
def løs_radiositet(T_overflate_K, F=F, emissivitet=emissivitet):
    Eps = np.diag(emissivitet)
    Rho = np.eye(len(emissivitet)) - Eps
    A_rad = np.eye(len(emissivitet)) - Rho @ F
    E_emit = Eps @ (sigma*T_overflate_K**4)

    J = ...
    G = ...
    Q = ...
    return J, Q

## C.3 ODE-systemet

Tilstanden lagres i Kelvin. Den fysiske temperaturvektoren for radiositetsberegningen settes sammen som

$$
(T_G,T_T,T_S,T_B,T_V).
$$

In [ ]:
def temperatur_ode(t, x):
    T_luft, T_gulv, T_glass = x

    T_overflate = np.array([
        T_gulv,
        T_faste_C["tak"] + 273.15,
        T_faste_C["side"] + 273.15,
        T_faste_C["bak"] + 273.15,
        T_glass
    ])

    J, Q_rad = løs_radiositet(T_overflate)
    Q_rad_gulv = Q_rad[G]
    Q_rad_glass = Q_rad[V]

    T_ute = T_ute_C(t) + 273.15

    dT_luft = (
        Q_romvarme(t)
        - h_tap*(T_luft - T_ute)
        - h_luft_gulv*A_G*(T_luft - T_gulv)
        - h_luft_glass*A_V*(T_luft - T_glass)
    )/C_luft

    dT_gulv = (
        Q_gulvvarme(t)
        + h_luft_gulv*A_G*(T_luft - T_gulv)
        - Q_rad_gulv
    )/C_gulv

    dT_glass = (
        U_glass*A_V*(T_ute - T_glass)
        + h_luft_glass*A_V*(T_luft - T_glass)
        - Q_rad_glass
    )/C_glass

    return np.array([dT_luft, dT_gulv, dT_glass])

## Oppgave C2: Implementer Euler

Simuler 24 timer. Begynn for eksempel med

$$T_{luft}=21^\circ C,\quad T_{gulv}=24^\circ C,\quad T_{glass}=12^\circ C.$$

Bruk først ett minutts tidssteg og kontroller deretter med et mindre steg.

In [ ]:
def euler_system(f, x0, sluttid, h):
    n = int(round(sluttid/h))
    t = np.linspace(0.0, n*h, n + 1)
    X = np.zeros((n + 1, len(x0)))
    X[0] = x0

    for k in range(n):
        X[k + 1] = ...

    return t, X

x0 = np.array([21.0, 24.0, 12.0]) + 273.15

t_C, X_C = euler_system(
    temperatur_ode,
    x0,
    sluttid=24*3600,
    h=60.0
)

plt.plot(t_C/3600, X_C[:, 0]-273.15, label="luft")
plt.plot(t_C/3600, X_C[:, 1]-273.15, label="gulv")
plt.plot(t_C/3600, X_C[:, 2]-273.15, label="glass")
plt.xlabel("Tid, timer")
plt.ylabel("Temperatur, grader C")
plt.grid()
plt.legend()
plt.show()

## Oppgave C3: Strålingsvarme gjennom døgnet

Beregn radiositet og netto strålingsvarme for hvert lagret tidspunkt. Plott særlig

- netto strålingsvarme fra gulvet,
- netto strålingsvarme til glassveggen,
- energibalansen i radiositetssystemet.

In [ ]:
Q_rad_historikk = []
balanse = []

for T_luft, T_gulv, T_glass in X_C:
    T_overflate = np.array([
        T_gulv,
        T_faste_C["tak"] + 273.15,
        T_faste_C["side"] + 273.15,
        T_faste_C["bak"] + 273.15,
        T_glass
    ])
    J, Q = løs_radiositet(T_overflate)
    Q_rad_historikk.append(Q)
    balanse.append(np.sum(Q))

Q_rad_historikk = np.array(Q_rad_historikk)
balanse = np.array(balanse)

# Lag plott.

## Oppgave C4: Sammenlign glassalternativer

Sammenlign minst to tilfeller, for eksempel

- lavere og høyere U-verdi,
- vanlig og lavemissiv innvendig glassflate,
- stor og liten varmekapasitet i glasset.

Vurder virkningen på

- innvendig glasstemperatur,
- varmestråling til glasset,
- lufttemperatur,
- antall iterasjoner i radiositetsløsningen dersom den iterative metoden brukes.

# Modellkritikk

Diskuter minst fem punkter:

- Hele frontveggen behandles som glass.
- Sideveggene antas helt symmetriske.
- Hver flate har uniform temperatur.
- Overflatene antas grå og diffuse.
- Luften absorberer og emitterer ikke langbølget stråling i modellen.
- Solinnstråling gjennom glasset er utelatt.
- Formfaktorene gjelder idealisert rektangulær geometri.
- Tak, sidevegger og bakvegg holdes faste i hoved-ODE-modellen.
- Konveksjonskoeffisientene er konstante.
- Lufttemperaturen er én samlet tilstand uten sjiktning.
- Materiallag, kuldebroer og varmelagring i alle konstruksjoner er ikke modellert.
- Iterasjonen representerer diffuse refleksjonsordener, ikke individuelle strålebaner.
- Euler kan kreve mindre tidssteg dersom modellen får raske tilstander.

## Mulige videreføringer

- et vanlig vindu som del av en vegg,
- tabellbaserte formfaktorer fra byggfaget,
- flere dynamiske overflatetemperaturer,
- en personflate og middelstrålingstemperatur,
- dagslysvariant med samme formfaktormatrise,
- solinnstråling gjennom glass,
- sammenligning med målte overflate- og lufttemperaturer,
- samarbeid med dataingeniører om automatisk matrisebygging og iterative løsere.

# Oppsummering

Skriv en kort rapport der du forklarer

1. hva en formfaktor betyr,
2. hvordan venstre og høyre sidevegg ble redusert til én radiositetsukjent,
3. hvorfor gulv og tak ble beholdt separat,
4. hvordan summasjon og resiprositet kontrollerte matrisen,
5. hvordan radiositetssystemet ble skrevet som $AJ=b$,
6. hvordan fortegnet til netto strålingsvarme ble tolket,
7. hvordan refleksjonsiterasjonen nærmet seg den direkte løsningen,
8. hvordan emissivitet påvirket refleksjoner og konvergens,
9. hvordan radiositetssystemet ble brukt inne i Euler-metoden,
10. hvorfor den kalde glassveggen påvirker både energibalanse og termisk komfort.

## Referanser for prosjektutviklingen

Prosjektet bygger på standard radiositetsmetode for grå, diffuse flater, standardreglene for formfaktorer og lukkede formler for rektangulære flater.

Studentene trenger ikke lese eksterne kilder for å gjennomføre prosjektet. Eventuelle formfaktortabeller fra byggfaget kan senere erstatte eller kontrollere hjelpefunksjonene i notebooken.